# 04 - Model Evaluation: 47-Class Character Performance

This notebook thoroughly evaluates model performance on 18,800 unseen test character images:
1. Evaluating test set accuracy and loss.
2. Generating a detailed classification report (Precision, Recall, F1-Score).
3. Visualizing the confusion matrix heatmap across all 47 classes (Numbers and Letters).
4. Analyzing character confusions (e.g. '0' vs 'O', '1' vs 'I', '8' vs 'B', '5' vs 'S').
5. Exporting evaluation summary metrics to `reports/evaluation_summary.json`.


In [ ]:
# Import libraries
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

sys.path.append(os.path.abspath(".."))
from src.preprocessing import load_preprocessed_data, load_label_mapping
from src.models import load_trained_model

print("Evaluation libraries loaded successfully!")


## 1. Load Model and Test Data


In [ ]:
features_path = os.path.abspath("../models/character_features.npz")
model_path = os.path.abspath("../models/handwritten_cnn.keras")
mapping_path = os.path.abspath("../models/label_mapping.json")

_, _, x_test, y_test = load_preprocessed_data(features_path)
model = load_trained_model(model_path)
label_map = load_label_mapping(mapping_path)


## 2. Evaluate Accuracy and Loss


In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=1)
print(f"\nFinal Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Final Test Loss:     {test_loss:.4f}")


## 3. Predictions and Classification Report


In [ ]:
y_pred_probs = model.predict(x_test, batch_size=128, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

class_names = [label_map[i] for i in range(len(label_map))]
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))


## 4. Confusion Matrix Heatmap


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    annot=False,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=True
)
plt.title(f"Confusion Matrix ({len(class_names)} Classes - Numbers & Letters)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Predicted Character Class", fontsize=12)
plt.ylabel("Actual Character Class", fontsize=12)
plt.xticks(rotation=90)
plt.yticks(rotation=0)

cm_path = os.path.abspath("../reports/confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Confusion matrix saved to: {cm_path}")


## Summary
- Evaluated on 18,800 test images spanning all numbers and letters.
- Confusion matrix and evaluation report saved in `reports/`.
